## 04 — Feature Importance: ML Techniques

Quantifies which features carry real predictive signal for `pitch_type` using two complementary approaches: correlation structure (to check redundancy within the `bat_*` block) and Mutual Information scoring (to compare compressed vs raw features).

**Sections:**
- Inter-stat correlation heatmap — mean absolute Pearson r across pitch types for every pair of `bat_*` stat families; reveals whether the 150-column block is effectively one signal or many
- PCA(5) loadings — what latent structure the principal components capture
- MI comparison — PC1–5 vs the top-10 individual `bat_*` features; settles whether compression unlocks collective signal that no single column carries alone

Run 02_data_quality and 03_pitch_distribution_eda first for context on data completeness and pitch mix intuition.

A high MI score means the feature (or component) reduces uncertainty about `pitch_type`. PCA loadings near ±1 on a stat family mean that family dominates the component; near 0 means it contributes little.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import pybaseball
from pybaseball import (
    cache,
    statcast,
    statcast_pitcher_pitch_arsenal,
    statcast_pitcher_arsenal_stats,
    statcast_batter_expected_stats,
    statcast_batter_pitch_arsenal,
)
from matplotlib.patches import Patch

sys.path.append(str(Path("..").resolve()))
from utils.features.feature_names import (
    TRAINABLE_COLUMNS,
    LABEL_COLUMN,
    STATCAST_PRE_PITCH,
    FANGRAPH_PRE_PITCH,
)

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

### Canonical pitch type codes (mirrors _PITCH_TYPES in feature_names.py)
PITCH_TYPES = ["FF", "SI", "FC", "SL", "CH", "CU", "FS", "KN", "ST", "SV"]

print("pybaseball :", pybaseball.__version__)
print("pandas     :", pd.__version__)
print("cache dir  :", cache.config.cache_directory)

pybaseball : 2.2.7
pandas     : 2.2.3
cache dir  : C:\Users\ihuang\Documents\python projects\MLB-Pitch-Predictor\data\cache


In [2]:
SEASON = 2024
PRIOR  = SEASON - 1

df = statcast("2024-06-01", "2024-06-30")
print(f"Raw Statcast shape: {df.shape}")

This is a large query, it may take a moment to complete


100%|██████████| 30/30 [00:00<00:00, 79.24it/s]


Raw Statcast shape: (116355, 118)


In [3]:
### Pitcher pitch-usage % (prior year)
pit_usage = statcast_pitcher_pitch_arsenal(PRIOR, arsenal_type="n_")
pit_usage_w = pit_usage.rename(columns={"pitcher_id": "pitcher"})  # col is already 'pitcher', no-op

### Pitcher per-pitch outcome stats (prior year)
pit_outcomes = statcast_pitcher_arsenal_stats(PRIOR, minPA=25)
pit_out_stat_cols = [
    c for c in pit_outcomes.columns
    if c not in ["last_name, first_name", "player_id", "team_name_alt", "pitch_type", "pitch_name"]
]
pit_outcomes_w = pit_outcomes.pivot_table(
    index="player_id", columns="pitch_type", values=pit_out_stat_cols, aggfunc="first"
)
pit_outcomes_w.columns = [f"{stat}_{pt}" for stat, pt in pit_outcomes_w.columns]
pit_outcomes_w = pit_outcomes_w.reset_index().rename(columns={"player_id": "pitcher"})

### Batter expected stats (prior year)
bat_xstats = statcast_batter_expected_stats(PRIOR)
bat_xstats_w = bat_xstats.rename(columns={"player_id": "batter"}).drop(
    columns=["last_name, first_name", "year"], errors="ignore"
)

### Batter vs pitch-type stats pivoted wide (prior year)
bat_vs_pitch = statcast_batter_pitch_arsenal(PRIOR, minPA=25)
bat_vs_stat_cols = [
    c for c in bat_vs_pitch.columns
    if c not in ["last_name, first_name", "player_id", "team_name_alt", "pitch_type", "pitch_name"]
]
bat_vs_pitch_w = bat_vs_pitch.pivot_table(
    index="player_id", columns="pitch_type", values=bat_vs_stat_cols, aggfunc="first"
)
bat_vs_pitch_w.columns = [f"bat_{stat}_{pt}" for stat, pt in bat_vs_pitch_w.columns]
bat_vs_pitch_w = bat_vs_pitch_w.reset_index().rename(columns={"player_id": "batter"})

### Merge all enrichment onto the pitch-level frame
df_enr = (
    df
    .merge(pit_usage_w,    on="pitcher", how="left", suffixes=("", "_pu"))
    .merge(pit_outcomes_w, on="pitcher", how="left", suffixes=("", "_po"))
    .merge(bat_xstats_w,   on="batter",  how="left", suffixes=("", "_bx"))
    .merge(bat_vs_pitch_w, on="batter",  how="left", suffixes=("", "_bv"))
)

print(f"Raw shape     : {df.shape}")
print(f"Enriched shape: {df_enr.shape}")
if "n_ff" in df_enr.columns:
    print(f"Pitcher arsenal coverage (n_ff non-null): {df_enr['n_ff'].notna().mean():.1%}")

Raw shape     : (116355, 118)
Enriched shape: (116355, 410)
Pitcher arsenal coverage (n_ff non-null): 75.0%


In [4]:
available    = [c for c in TRAINABLE_COLUMNS if c in df_enr.columns]
missing_cols = [c for c in TRAINABLE_COLUMNS if c not in df_enr.columns]

model_df = df_enr[available + [LABEL_COLUMN]].copy()
model_df = model_df[model_df[LABEL_COLUMN].isin(PITCH_TYPES)].reset_index(drop=True)

print(f"model_df shape          : {model_df.shape}")
print(f"TRAINABLE_COLUMNS total : {len(TRAINABLE_COLUMNS)}")
print(f"Found in enriched frame : {len(available)}")
if missing_cols:
    preview = missing_cols[:5]
    ellipsis = "..." if len(missing_cols) > 5 else ""
    print(f"Missing ({len(missing_cols)})            : {preview}{ellipsis}")
print(f"\nPitch type distribution:\n{model_df[LABEL_COLUMN].value_counts()}")

model_df shape          : (114080, 323)
TRAINABLE_COLUMNS total : 352
Found in enriched frame : 322
Missing (30)            : ['bat_run_value_per_100_KN', 'bat_run_value_per_100_SV', 'bat_run_value_KN', 'bat_run_value_SV', 'bat_pitches_KN']...

Pitch type distribution:
pitch_type
FF    37126
SI    18600
SL    16954
CH    11444
FC     9449
ST     8264
CU     7663
FS     3747
SV      563
KN      270
Name: count, dtype: int64
